In [1]:
import pyaudio
import struct
import numpy as np
import matplotlib.pyplot as plt

# Apply a magic command to allow tinker being used with matplotlib
# This allows for visualizations in seperate windows
%matplotlib tk 

CHUNK = 1024 * 4    # number of audio samples to collect       
FORMAT = pyaudio.paInt16  # a digitized audio sample is represented by 16-bit value
CHANNELS = 1    # mono audio 
RATE = 44100    # sampling rate = 44.1 kHz 

### 1- REAL-TIME AUDIO SIGNAL PRESENTATION

In [2]:
# initialize an audio system
p = pyaudio.PyAudio()

# open and audio stream with the specified parameters
#  a stream is a buffer to save the real-time digital data, which is 
#  managed by OS,  rightafter it is collected from the soundcard
# In addition to the configuration task, the command "open" also perform
# a sound capture and save into the "stream" object.  
stream = p.open(format = FORMAT,    
                channels = CHANNELS,
                rate = RATE,
                input = True,   # allow to receive sound from the microphone
                output = False,  # allow to send sound to the speaker
                frames_per_buffer = CHUNK)

# data saved in stream is raw binary data which in the format of a single byte
# data = stream.read(CHUNK)
# print(len(data))

# the struct library to convert between python-defined type and raw binary type

fig, ax = plt.subplots()

x = np.arange(0,  CHUNK, 1)
line, = ax.plot(x,np.random.rand(CHUNK))
ax.set_ylim(-20000,20000)
ax.set_xlim(0, CHUNK)

while True:
    try:
        data = stream.read(CHUNK, exception_on_overflow=False)
        data_int = np.frombuffer(data, dtype=np.int16)
        line.set_ydata(data_int) 
        fig.canvas.draw()
        fig.canvas.flush_events()
    except IOError as e:
        print(f"Overflow error: {e}")
        continue


### 2 -  RECORD AUDIO AND PRESENT THE SIGNAL 

In [4]:
# Audio configuration
FORMAT = pyaudio.paInt16  # 16-bit resolution
CHANNELS = 1              # Mono audio
RATE = 44100              # Sampling rate (44.1 kHz)
CHUNK = 1024              # Number of samples per buffer
RECORD_SECONDS = 5        # Duration of recording
DEVICE_INDEX = 1          # Change if you want to select a specific input device

# Initialize pyaudio
audio = pyaudio.PyAudio()

# Start the microphone stream
stream = audio.open(format=FORMAT, channels=CHANNELS,
                    rate=RATE, input=True,
                    frames_per_buffer=CHUNK)

print("Recording...")

# Record audio data
frames = []
for _ in range(int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK)
    frames.append(np.frombuffer(data, dtype=np.int16))

# Stop recording
stream.stop_stream()
stream.close()
audio.terminate()

# Convert the recorded frames into a numpy array
audio_signal = np.hstack(frames)

# Time values for plotting
time = np.linspace(0, RECORD_SECONDS, num=len(audio_signal))

# Plot the sound signal
plt.figure(figsize=(10, 5))
plt.plot(time, audio_signal, label="Audio Signal")
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.title("Sound Signal from Microphone")
plt.grid()
plt.show()


In [5]:
from scipy.fft import fft, fftfreq

In [6]:
y_fft = fft(audio_signal)
y_freq = fftfreq(len(audio_signal))

plt.plot(y_freq, np.abs(y_fft))

### SPECTRUM ANALYZER 



In [ ]:
import numpy as np
import pyaudio
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq

# ======================
# Audio config
# ======================
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 44100
CHUNK = 1024

# ======================
# PyAudio
# ======================
p = pyaudio.PyAudio()
stream = p.open(
    format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK
)

plt.ion()

# ======================
# Create 4 subplots
# ======================
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 6))

x_time = np.arange(CHUNK)
x_freq = fftfreq(CHUNK, 1 / RATE)

# 1️⃣ Time - Original
(line1,) = ax1.plot(x_time, np.zeros(CHUNK))
ax1.set_title("Original Signal")
ax1.set_ylim(-20000, 20000)

# 2️⃣ Spectrum - Original
(line2,) = ax2.plot(x_freq, np.zeros(CHUNK))
ax2.set_title("Original Spectrum")
ax2.set_ylim(0, 1)

# 3️⃣ Time - Filtered
(line3,) = ax3.plot(x_time, np.zeros(CHUNK))
ax3.set_title("After Removing 1000Hz")
ax3.set_ylim(-20000, 20000)

# 4️⃣ Spectrum - Filtered
(line4,) = ax4.plot(x_freq, np.zeros(CHUNK))
ax4.set_title("Spectrum without 1000Hz")
ax4.set_ylim(0, 1)

# ======================
# Real-time loop
# ======================
while True:
    try:
        sound_sig = stream.read(CHUNK, exception_on_overflow=False)
        sound_data = np.frombuffer(sound_sig, dtype=np.int16)

        # ===== FFT original =====
        X = fft(sound_data)
        spectrum_original = np.abs(X)

        # Normalize original spectrum
        max_val = np.max(spectrum_original)
        if max_val != 0:
            spectrum_original = spectrum_original / max_val

        # ===== Remove 1000 Hz =====
        freqs = fftfreq(len(sound_data), 1 / RATE)
        idx_1000 = np.argmin(np.abs(freqs - 1000))

        X_filtered = X.copy()
        X_filtered[idx_1000] = 0
        X_filtered[-idx_1000] = 0

        # ===== iFFT =====
        filtered_signal = np.fft.ifft(X_filtered).real
        filtered_signal = np.clip(filtered_signal, -32768, 32767)
        filtered_signal = filtered_signal.astype(np.int16)

        # ===== Spectrum filtered =====
        spectrum_filtered = np.abs(X_filtered)
        max_val_f = np.max(spectrum_filtered)
        if max_val_f != 0:
            spectrum_filtered = spectrum_filtered / max_val_f

        # ===== Update plots =====
        line1.set_ydata(sound_data)
        line2.set_ydata(spectrum_original)

        line3.set_ydata(filtered_signal)
        line4.set_ydata(spectrum_filtered)

        fig.canvas.draw()
        fig.canvas.flush_events()

    except IOError as e:
        print(f"overflow error: {e}")
        continue

In [5]:
import numpy as np
import pyaudio
import matplotlib.pyplot as plt
from scipy import signal

%matplotlib tk

# ======================
# Audio parameters
# ======================
FORMAT   = pyaudio.paInt16
CHANNELS = 1
RATE     = 44100
CHUNK    = 1024

# ======================
# FIR design (example: low-pass)
# ======================
numtaps = 257
cutoff_hz = 3000

b = signal.firwin(
    numtaps,
    cutoff_hz,
    fs=RATE,
    window="hamming"
)

# FIR state (important for streaming)
zi = np.zeros(len(b) - 1, dtype=np.float32)

# ======================
# PyAudio setup
# ======================
p = pyaudio.PyAudio()
stream = p.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)

# ======================
# Matplotlib setup
# ======================
plt.ion()

x = np.arange(CHUNK)

# --- Window 1: Input ---
fig_in, ax_in = plt.subplots()
line_in, = ax_in.plot(x, np.zeros(CHUNK))
ax_in.set_title("Microphone Input")
ax_in.set_ylim(-20000, 20000)
ax_in.set_xlim(0, CHUNK)

# --- Window 2: Output ---
fig_out, ax_out = plt.subplots()
line_out, = ax_out.plot(x, np.zeros(CHUNK))
ax_out.set_title("FIR Filtered Output")
ax_out.set_ylim(-20000, 20000)
ax_out.set_xlim(0, CHUNK)

# ======================
# Real-time loop
# ======================
while True:
    data = stream.read(CHUNK, exception_on_overflow=False)

    # raw input
    x_int16 = np.frombuffer(data, dtype=np.int16)
    x_f = x_int16.astype(np.float32)

    # FIR filtering (streaming)
    y_f, zi = signal.lfilter(b, [1.0], x_f, zi=zi)

    y_int16 = np.clip(y_f, -32768, 32767).astype(np.int16)

    # update plots
    line_in.set_ydata(x_int16)
    line_out.set_ydata(y_int16)

    fig_in.canvas.draw()
    fig_in.canvas.flush_events()

    fig_out.canvas.draw()
    fig_out.canvas.flush_events()
